# Arctic Pollinator Detection Pipeline

## Pipeline Overview

1. **Sort & preprocess** — Sort by EXIF timestamp. Skip flash (night) and foggy frames. Remove 120px camera info strip from bottom. Classify weather from shutter speed.
2. **Background model** — Default: frame-to-frame difference (`rolling_window=1`). Alternative: global median background (`background_sample_size > 0`).
3. **ROI detection** — Detect coloured marker clip via HSV segmentation. Build circular zone around it. Falls back to full image if no marker found.
4. **Foreground detection** — Absolute diff vs background, brightness-normalised, Gaussian-blurred, thresholded. Green vegetation masked out. Morphological open/close to clean noise. Contours filtered by area, aspect ratio, and size. Nearby boxes merged.
5. **Large motion handling** — Large foreground regions (flower displaced by bumblebee) are split into overlapping tiles and lower-region fallback crops to avoid missing the insect.
6. **Static filter** — Detections at the same position for >N frames are flagged (`static_suspect=True`) or dropped depending on `static_filter_mode`.
7. **Two-stage classification**

   - *Binary classifier (EfficientNet-B2):* insect vs background. Background crops are discarded — no file saved, no CSV row written.
   - *Group classifier (InsectNet backbone):* bumblebee / fly / butterfly_moth / other. Only runs on crops confirmed as insect.
   - Set `skip_classification=True` to skip both and save all candidate crops.
8. **Output** — One CSV row per confirmed insect crop. Crop saved to `crops/`. Debug overlays saved to `debug/` for all frames regardless of classification result.

---

## Output Structure

```
results_N/
└── camera_folder/
    ├── results.csv
    ├── crops/
    └── debug/
```

**Crop filename:** `{camera_path}__{image_stem}_crop_{i}_{type}_{roi/out}.jpg`

---

## Debug Image Colours

| Colour | Meaning                     |
| ------ | --------------------------- |
| Green  | Inside ROI, passes filters  |
| Blue   | Outside ROI, passes filters |
| Red    | Filtered out                |
| Yellow | Static suspect inside ROI   |
| Pink   | Static suspect outside ROI  |
| Orange | ROI boundary                |

---

## Key Config Parameters

| Parameter                | Default   | Effect                                              |
| ------------------------ | --------- | --------------------------------------------------- |
| `rolling_window`       | 1         | 1 = frame-to-frame diff                             |
| `skip_classification`  | False     | True = save all candidate crops without classifying |
| `binary_threshold`     | 0.5       | Min confidence to keep as insect                    |
| `use_roi`              | True      | False = detect on full image                        |
| `crop_mode`            | `"all"` | `"roi_only"` or `"out_only"`                    |
| `enable_static_filter` | False     | Toggle static suspect pass                          |
| `strip_height`         | 120       | Pixels to remove from bottom                        |
| `debug_outputs`        | `"all"` | `"annotate"` = only final overlay                 |
| `frames_from`          | None      | Path to triage file (one frame per line)            |

---

## Known Limitations

- Stationary insects are missed (absorbed into background)
- Camouflaged insects are missed (colour too similar to vegetation)
- Camera angle changes mid-sequence invalidate the ROI — split affected sequences manually before running
- Visit count is approximate — one crop per frame, so a single insect across multiple frames is counted multiple times

### Imports

In [1]:
import sys
import csv
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torchvision
from PIL import Image as PILImage
from PIL.ExifTags import TAGS


# Resolve notebook directory robustly — works even if kernel CWD differs
_nb_dir = Path(globals().get("__vsc_ipynb_file__", "")).parent
if not _nb_dir.is_dir():
    _nb_dir = Path.cwd()
sys.path.insert(0, str(_nb_dir / "InsectNet"))

PROJECT_DIR = _nb_dir

# Folder with all images
IMAGE_FOLDER = "Insects_images/Pollinators"

### Configuration

All tunable parameters are here. Override any of them when calling `main()` without touching the rest of the code.

Parameters marked **TUNE** are the ones most likely to need adjustment per dataset or plot.

In [2]:
DEFAULT_CONFIG = {

    # Background computation
    # Number of frames to sample for median background. TUNE
    # Set to 0 or None to skip global background entirely (use rolling_window instead).
    # Set to 0 + rolling_window > 0 = pure rolling/frame-to-frame mode (no memory overhead).
    "background_sample_size":    0,

    # Background subtraction
    # How much darker/lighter than the median a pixel must be to count as foreground.
    # Lower = more sensitive (more detections, more noise).
    # Higher = less sensitive (fewer detections, fewer false positives). TUNE
    "darker_threshold":          30,   # detect colour-similar insects (yellow/black)

    # Contour filtering
    "min_contour_area":          400,   # small flies can be tiny, but soil speckles are usually smaller. TUNE
    "max_contour_area":          35000,  # normal contour upper limit
    "max_large_motion_area":     600000, # large plant-motion regions are retained, not discarded
    "large_motion_tile_sizes":   [320, 512], # multi-scale tiles with fewer redundant crops
    "large_motion_tile_stride_frac": 0.65,  # larger stride reduces near-duplicate tiles
    "large_motion_max_tiles_per_size": 6,   # cap tiles per scale after scoring and NMS
    "large_motion_max_tiles_total": 10,      # total tile cap per large motion region
    "large_motion_tile_nms_iou": 0.35,    # remove overlapping/redundant tiles
    "large_motion_min_fg_frac":  0.008,  # tile must contain at least this foreground fraction
    "large_motion_context_pad":  10,     # small padding for full large-motion context crop
    "large_motion_fallback_sizes": [640], # lower-region crops for insects at the edge
    "large_motion_fallback_centers": [
        (0.35, 0.78), (0.50, 0.78), (0.65, 0.78),
    ],
    "large_motion_max_fallbacks": 3,
    "large_motion_fallback_min_fg_frac": 0.003,
    "min_crop_px":               10,     # very low — don't filter small flies, padding handles size
                                         # boxes smaller than this after merging are skipped
                                         # (too small for any classifier to recognise)
    # Max pixel gap between boxes to merge — 20px catches split fly body parts
    # without merging two separate insects (usually >20px apart)
    "merge_dist":        20,
    "max_aspect_ratio":          5,    # raised to catch elongated insects
    "kernel_open_size":          3,    # morphological open kernel — removes isolated speckles
    "kernel_close_size":         11,   # morphological close kernel — merges insect body fragments. TUNE
    # Minimum grayscale std dev inside contour (insects have texture, soil does not). TUNE
    "min_texture":               50,

    # Static detection removal
    # Toggle the whole pass on/off without changing thresholds — useful for
    # A/B comparisons of how much the static filter is doing.
    "enable_static_filter":      False,
    "static_dist":               80,
    "static_max_frames":         15,
    "static_filter_mode":        "flag", # "flag" keeps crops; "drop" removes static suspects

    # Crop padding (proportional — 10% of bbox size)
    # Crop window sizing — see crop_with_padding() for details
    # Tune these multipliers if insects are too small/large in saved crops:
    #   small_fly_multiplier:  2.5  (eff < 50px)
    #   medium_multiplier:     1.8  (50 <= eff < 120px)
    #   large_multiplier:      1.2  (eff >= 120px)
    # Window is always centred on bbox centre, not bbox corner.

    # Rolling background window. TUNE
    # 0 = use global median background (default)
    # 1 = pure frame-to-frame diff (no memory, misses stationary insects)
    # N > 1 = median of last N frames (bumblebee visible if it moves within N frames)
    # Tip: set background_sample_size=0 to skip building global background entirely
    "rolling_window": 1,

    # Marker / ROI
    "marker_hue":                (45, 75),
    "marker_sat_min":            200,
    "marker_val_min":            100,
    "marker_min_area":           200,
    "marker_zone_radius":        800,

    # Detection-loop progress: print a status line every N frames (0 = silent).
    "progress_every":            50,

    # Quality filtering
    "skip_flash":                True,   # skip frames where flash fired (night shots). TUNE
    "skip_foggy":                True,   # skip frames below foggy_threshold. TUNE
    "foggy_threshold":           50,    # Laplacian variance below this = fog/blur. TUNE

    # Weather classification from EXIF shutter speed
    "sunny_shutter_threshold":   150,

    # Near-flower detection
    "near_flower_iou_threshold": 0.1,

    # Pollinator classification uses a two-stage pipeline:
    # Stage 1 (binary): insect vs background
    # Stage 2 (group): bumblebee / fly / butterfly_moth / other

    # Manual triage: when set, only process the image paths listed in this
    # file (one path per line). Path is resolved relative to image_dir if
    # not absolute. Off by default — pair with feh-triage.sh to flag frames.
    "frames_from":               None,         # e.g. "flagged.txt"; None = all images
    # Around each flagged frame, also include this many neighbours on each
    # side (rolling-bg context). 0 = strict, only the listed frames.
    "frames_context":            3,

    # Sort frames by EXIF DateTimeOriginal instead of filename. Slow — opens
    # every JPG. Default False is correct for sequential-filename cameras
    # (e.g. Wingscapes) where one leaf folder = one session. Set True only if
    # filenames may not reflect capture order (merged dumps from multiple
    # cameras, WSCT9999 rollover within a single folder).
    "use_exif_sort": False,

    # Manual ROI — set True to draw ROI manually instead of auto-detecting marker
    # Useful for cameras where the colored marker is not visible
    "manual_roi": False,
      "use_roi": True,   # False = full image detection

      # Bottom info strip — the camera burns timestamp/temp/camera info into the
      # bottom of every image. Set strip_height to the pixel height of that bar.
      # strip_height=0 disables cropping. The strip is cropped BEFORE detection
      # so text pixels cannot trigger false positives.
      "strip_height":          120,   # pixels to crop from bottom — matches Wingscapes TLCAM PRO info bar
      # OCR is the slowest step in the loop (≈0.3s/frame via tesseract subprocess
      # spawn) and the temperature column isn't currently consumed downstream.
      # Off by default; flip to True if a downstream consumer needs the values.
      "strip_ocr_temperature": False,

      # crop_mode controls which detections are saved as crops:
      #   "all"      → save all detections (inside + outside ROI)
      #   "roi_only" → save only detections inside ROI
      #   "out_only" → save only detections outside ROI
      "crop_mode": "all",

    # Set to True to skip classification — only save crops (useful for tuning detection).
    "skip_classification": False,

    # Binary classifier threshold — crops above this confidence are classified as insect.
    # Lower = higher recall (more false positives); Higher = higher precision.
    "binary_threshold": 0.5,

    # Debug image saving
    # debug_outputs:
    #   "all"      -> save _1_original, _2_diff, _3_contours, _4_final_saved_crops
    #   "annotate" -> save only _4_final_saved_crops + _4_offset.json
    #   list/set   -> any of ["original", "diff", "contours", "final"]
    "debug_outputs": "all",
    "debug_save_empty_frames": True,
    "debug_max_width": None,    # downscale debug images only; crops stay full quality
    "debug_max_height": None,
    "debug_jpeg_quality": 90,
}

# ── Developer CSV: full technical detail for debugging and model development ──
CSV_FIELDS_DEBUG = [
    # Identity — where it came from
    "camera_path",          # camera identifier (parent_leaf folder) — FIRST for easy sorting
    "image_name",           # source image filename
    "crop_filename",        # saved crop filename (globally unique)
    "datetime",             # EXIF timestamp
    "temperature_c",        # temperature OCR from info strip
    "camera_name",          # camera model from EXIF

    # Image quality / filtering
    "skip",                 # True if image was skipped
    "skip_reason",          # reason for skip (flash / fog / etc.)
    "laplacian_var",        # sharpness score (lower = blurrier)
    "shutter_speed",        # e.g. 1/500
    "weather",              # sunny / cloudy (derived from shutter speed)

    # Detection result
    "pollinator_detected",  # candidate / no / skipped / uncertain
    "crop_filename",        # crop file (duplicate key removed below)
    "bbox_x", "bbox_y", "bbox_w", "bbox_h",   # bounding box pixels
    "detection_scope",      # roi / outside_roi
    "near_marked_flower",   # True / False
    "candidate_type",       # normal / large_motion_context / large_motion_tile
    "static_suspect",       # True if in same position for >N consecutive frames
    "source_area",          # source bbox for tiles

    # Two-stage classifier results
    "binary_label",         # insect / background
    "binary_confidence",    # binary classifier confidence 0-1
    "pollinator_type",      # bumblebee / fly / butterfly_moth / other (empty if binary=background)
    "group_confidence",     # group classifier confidence 0-1
    "bumblebee_prob",       # per-class probability
    "fly_prob",
    "butterfly_moth_prob",
    "other_prob",
]

# De-duplicate (crop_filename appeared twice above by mistake)
CSV_FIELDS_DEBUG = list(dict.fromkeys(CSV_FIELDS_DEBUG))

# ── Researcher CSV: clean output for ecological analysis ──────────────────────
CSV_FIELDS_MARIA = [
    # Identity — what image, from which camera
    "camera_path",          # Camera location identifier — FIRST for easy grouping
    "image_name",           # Image filename (for cross-reference with raw data)
    "crop_filename",        # Cropped region filename

    # When & conditions
    "datetime",             # Date and time of image
    "temperature_c",        # Temperature (deg C) from camera strip
    "weather",              # sunny / cloudy

    # Detection result
    "pollinator_detected",  # candidate / no
    "near_marked_flower",   # True = near focal plant, False = elsewhere in frame
    "detection_scope",      # roi = near focal plant / outside_roi = rest of image

    # Bounding box — pixel coords in original image (needed for annotation tool)
    "bbox_x", "bbox_y", "bbox_w", "bbox_h",

    # Classification results
    "pollinator_type",      # bumblebee / fly / butterfly_moth / other
    "binary_confidence",    # how certain the model is that this is an insect (0-1)
    "group_confidence",     # how certain the model is of the pollinator type (0-1)
]


### Helper functions

In [3]:

from pathlib import Path

def get_relative_path(path, root):
    return Path(path).relative_to(root)

def ensure_dir(p):
    p.mkdir(parents=True, exist_ok=True)
import re
from PIL import Image as PILImageSort
from PIL.ExifTags import TAGS as TAGS_SORT

# Find EXIF DateTimeOriginal tag ID
_EXIF_DT_TAG = next((k for k, v in TAGS_SORT.items() if v == "DateTimeOriginal"), None)
_exif_cache: dict = {}

def get_exif_datetime_sort(path) -> str | None:
    """Read EXIF DateTimeOriginal from image, with caching."""
    key = str(path)
    if key in _exif_cache:
        return _exif_cache[key]
    try:
        img = PILImageSort.open(path)
        exif = img._getexif() if hasattr(img, "_getexif") else dict(img.getexif())  # type: ignore[attr-defined]
        val = exif.get(_EXIF_DT_TAG) if exif and _EXIF_DT_TAG else None
    except Exception:
        val = None
    _exif_cache[key] = val
    return val

def _parse_exif_dt(dt_str: str):
    """Parse '2023:07:14 12:34:56' into a sortable tuple."""
    try:
        date, time = dt_str.split(" ")
        y, m, d = map(int, date.split(":"))
        hh, mm, ss = map(int, time.split(":"))
        return (y, m, d, hh, mm, ss)
    except Exception:
        return None

def robust_sort_key(p):
    """Sort by: 1) EXIF DateTimeOriginal, 2) filename number, 3) filename string.

    Slow — opens every JPG to read EXIF. Use only when filenames may not
    reflect capture order (merged camera dumps, WSCT9999 rollover, etc.).
    """
    from pathlib import Path
    p = Path(p)
    exif_dt = get_exif_datetime_sort(p)
    if exif_dt:
        parsed = _parse_exif_dt(exif_dt)
        if parsed:
            return (0, parsed, p.name)
    m = re.search(r'(\d+)', p.stem)
    if m:
        return (1, int(m.group(1)), p.name)
    return (2, p.name)


def filename_sort_key(p):
    """Sort by numeric suffix of filename (no file I/O — cheap).

    Correct for cameras that name files sequentially within one session
    (e.g. WSCT0001..WSCT9999) when one leaf folder is one session.
    """
    from pathlib import Path
    p = Path(p)
    m = re.search(r'(\d+)', p.stem)
    if m:
        return (0, int(m.group(1)), p.name)
    return (1, p.name)

# ══════════════════════════════════════════════════════════════════
# ROI / ZONE
# ══════════════════════════════════════════════════════════════════

def find_marker(image, cfg):
    """Find the coloured marker clip in the image. Returns centroid (cx, cy) or None."""
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(
        hsv,
        np.array([cfg["marker_hue"][0], cfg["marker_sat_min"], cfg["marker_val_min"]]),
        np.array([cfg["marker_hue"][1], 255, 255]),
    )
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    clusters = [c for c in contours if cv2.contourArea(c) > cfg["marker_min_area"]]
    if not clusters:
        return None
    h_img, w_img = image.shape[:2]
    cx_img, cy_img = w_img // 2, h_img // 2

    def dist_to_center(c):
        M = cv2.moments(c)
        if M["m00"] == 0:
            return float("inf")
        return (int(M["m10"]/M["m00"]) - cx_img)**2 + (int(M["m01"]/M["m00"]) - cy_img)**2

    best = min(clusters, key=dist_to_center)
    M = cv2.moments(best)
    return int(M["m10"] / M["m00"]), int(M["m01"] / M["m00"])


def build_marker_zone(image, marker, cfg):
    """Create a circular binary mask (zone) around the marker position."""
    h_img, w_img = image.shape[:2]
    zone = np.zeros((h_img, w_img), dtype=np.uint8)
    cv2.circle(zone, marker, cfg["marker_zone_radius"], 255, -1)
    return zone


def select_roi(image_path):
    """Open the first image and let user draw a rectangle as the watching zone."""
    img = cv2.imread(image_path)
    h, w = img.shape[:2]
    scale = min(1.0, 1200 / w)
    display = cv2.resize(img, (int(w * scale), int(h * scale)))
    print("Draw a rectangle around the flower area, then press ENTER or SPACE.")
    roi = cv2.selectROI("Select flower region", display, showCrosshair=True)
    cv2.destroyAllWindows()
    x, y, rw, rh = [int(v / scale) for v in roi]
    if rw == 0 or rh == 0:
        return None
    zone = np.zeros((h, w), dtype=np.uint8)
    zone[y:y+rh, x:x+rw] = 255
    print(f"ROI selected: x={x} y={y} w={rw} h={rh}")
    return zone


def setup_roi(paths, cfg, manual_roi=False):
    """Define ROI from green marker, manual selection, or full image fallback.

    Priority:
    1. manual_roi=True -> draw ROI manually
    2. Auto-detect coloured marker -> build circular zone
    3. No marker found -> fallback to full image
    """
    first_img = cv2.imread(str(paths[0]))

    if not cfg.get("use_roi", True):
        print("use_roi=False — detecting on full image")
        zone   = np.ones(first_img.shape[:2], dtype=np.uint8) * 255
        marker = find_marker(first_img, cfg)
        if marker:
            print(f"Marker found at ({marker[0]}, {marker[1]}) — used for ROI labeling only")
        return zone, marker

    if manual_roi:
        zone = select_roi(str(paths[0]))
        if zone is None:
            print("No region selected — falling back to full image as ROI.")
            zone = np.ones(first_img.shape[:2], dtype=np.uint8) * 255
        return zone, None

    marker = find_marker(first_img, cfg)
    if marker is None:
        print("No marker found — using full image as ROI (set manual_roi=True to draw manually)")
        zone = np.ones(first_img.shape[:2], dtype=np.uint8) * 255
        return zone, None

    print(f"Marker found at ({marker[0]}, {marker[1]})")
    zone = build_marker_zone(first_img, marker, cfg)
    pct = 100 * np.count_nonzero(zone) / zone.size
    print(f"Watching zone covers {pct:.1f}% of image")
    return zone, marker


In [4]:
# ══════════════════════════════════════════════════════════════════
# MODEL — Two-stage classifier pipeline
#
# Stage 1 — Binary classifier (EfficientNet-B2 or InsectNet backbone):
#   Distinguishes insect crops from background crops.
#   Trained on Arctic field crops from two labeled subsets.
#
# Stage 2 — Group classifier (InsectNet backbone, Stage 1 only):
#   Assigns confirmed insect crops to one of four categories:
#   bumblebee / fly / butterfly_moth / other
#   Trained on Arctic crops + iNaturalist web images (Sweden/Norway).
# ══════════════════════════════════════════════════════════════════

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T
from PIL import Image as _PILClassify

BINARY_CLASSES = ["background", "insect"]
GROUP_CLASSES  = ["bumblebee", "fly", "butterfly_moth", "other"]


def _letterbox(img, size):
    w, h  = img.size
    max_s = max(w, h)
    sq    = _PILClassify.new("RGB", (max_s, max_s), (0, 0, 0))
    sq.paste(img, ((max_s - w) // 2, (max_s - h) // 2))
    return sq.resize((size, size), _PILClassify.BILINEAR)


def _make_transform(img_size):
    return T.Compose([
        T.Lambda(lambda img: _letterbox(img, img_size)),
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])


def _build_efficientnet_b2(n_classes):
    model = torchvision.models.efficientnet_b2(weights=None)
    in_f  = model.classifier[-1].in_features
    model.classifier[-1] = nn.Linear(in_f, n_classes)
    return model


def _build_insectnet_backbone(n_classes):
    model    = torchvision.models.regnet_y_32gf()
    model.fc = nn.Linear(3712, n_classes)
    return model


def load_binary_classifier(ckpt_path, device=None):
    """Load trained binary insect/background classifier."""
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ckpt   = torch.load(ckpt_path, map_location=device, weights_only=False)
    model_name = ckpt.get("model", "efficientnet")
    img_size   = ckpt.get("img_size", 256)
    if "insectnet" in model_name.lower():
        model = _build_insectnet_backbone(2)
    else:
        model = _build_efficientnet_b2(2)
    model.load_state_dict(ckpt["state_dict"])
    model.eval()
    print(f"Binary classifier loaded: {model_name} | img={img_size}px | device={device}")
    return model.to(device), _make_transform(img_size), device


def load_group_classifier(ckpt_path, device=None):
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ckpt   = torch.load(ckpt_path, map_location=device, weights_only=False)
    img_size = ckpt.get("img_size", 224)
    classes  = ckpt.get("classes", GROUP_CLASSES)

    # Auto-detect architecture from state_dict keys
    state_keys = list(ckpt["state_dict"].keys())
    if any("trunk_output" in k or "stem." in k for k in state_keys):
        arch  = "insectnet"
        model = _build_insectnet_backbone(len(classes))
    else:
        arch  = "efficientnet"
        model = _build_efficientnet_b2(len(classes))

    model.load_state_dict(ckpt["state_dict"])
    model.eval()
    print(f"Group classifier loaded: {arch} | classes={classes} | img={img_size}px")
    return model.to(device), _make_transform(img_size), classes, device


def classify_crop(crop_bgr, binary_model, binary_tf, group_model, group_tf,
                  group_classes, device, binary_threshold=0.5):
    """
    Run two-stage classification on one crop (BGR numpy array).

    Returns dict with keys:
        binary_label, binary_confidence,
        pollinator_type, group_confidence,
        bumblebee_prob, fly_prob, butterfly_moth_prob, other_prob
    """
    # Convert BGR → RGB → PIL
    crop_rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)
    img      = _PILClassify.fromarray(crop_rgb)

    # Stage 1: binary
    x = binary_tf(img).unsqueeze(0).to(device)
    with torch.no_grad():
        probs_bin = torch.softmax(binary_model(x), dim=1)[0]
    bin_idx  = probs_bin.argmax().item()
    bin_conf = float(probs_bin[bin_idx])
    bin_label = BINARY_CLASSES[bin_idx]

    result = {
        "binary_label":      bin_label,
        "binary_confidence": round(bin_conf, 4),
        "pollinator_type":   "",
        "group_confidence":  "",
        "bumblebee_prob":    "",
        "fly_prob":          "",
        "butterfly_moth_prob": "",
        "other_prob":        "",
    }

    # Stage 2: group (only if binary says insect)
    if bin_label == "insect" and bin_conf >= binary_threshold:
        x2 = group_tf(img).unsqueeze(0).to(device)
        with torch.no_grad():
            probs_grp = torch.softmax(group_model(x2), dim=1)[0]
        grp_idx   = probs_grp.argmax().item()
        grp_label = group_classes[grp_idx]
        grp_conf  = float(probs_grp[grp_idx])
        all_probs = {c: float(probs_grp[i]) for i, c in enumerate(group_classes)}

        result["pollinator_type"]      = grp_label
        result["group_confidence"]     = round(grp_conf, 4)
        result["bumblebee_prob"]       = round(all_probs.get("bumblebee", 0), 4)
        result["fly_prob"]             = round(all_probs.get("fly", 0), 4)
        result["butterfly_moth_prob"]  = round(all_probs.get("butterfly_moth", 0), 4)
        result["other_prob"]           = round(all_probs.get("other", 0), 4)

    return result


def load_model_and_classes():
    """Load both classifiers. Called once before main()."""
    print("Loading classifiers...")
    binary_model, binary_tf, device = load_binary_classifier(
        PROJECT_DIR / "models" / "binary_best.pth"
    )
    group_model, group_tf, group_classes, _ = load_group_classifier(
        PROJECT_DIR / "models" / "group_best.pth", device=device
    )
    print("Classifiers loaded.\n")
    # Return in same signature as before: (binary_model, group_model, group_tf, group_classes, binary_tf, device)
    return binary_model, binary_tf, group_model, group_tf, group_classes, device


In [5]:
# ══════════════════════════════════════════════════════════════════
# MODEL
# ══════════════════════════════════════════════════════════════════

# ══════════════════════════════════════════════════════════════════
# ROI / ZONE
# ══════════════════════════════════════════════════════════════════
# ══════════════════════════════════════════════════════════════════
# BACKGROUND
# ══════════════════════════════════════════════════════════════════
def build_background(paths, cfg):
    """
    Step 2: Compute median background from a uniform sample of frames.

    Uses cfg['background_sample_size'] frames evenly spaced across the sequence.
    Avoids loading all 12,000 frames into memory (~60GB) while producing
    a background equivalent to the full-dataset median.
    Per-pixel median naturally excludes transient objects like insects (~4% of frames).
    """
    n = cfg["background_sample_size"]
    # background_sample_size=0/None, skip global background.
    # rolling window in detect_all_frames handles all background subtraction.
    if not n:
        print("background_sample_size=0 — skipping global background (rolling only)")
        return None
    step = max(1, len(paths) // n)
    sampled = list(paths)[::step][:n]
    frames = [cv2.imread(str(p)) for p in sampled]
    frames = [f for f in frames if f is not None]
    if not frames:
        return None
    print(f"Background computed from {len(frames)} sampled frames (of {len(paths)} total)")
    return np.median(frames, axis=0).astype(np.uint8)


# ══════════════════════════════════════════════════════════════════
# DETECTION (background subtraction + contour filtering)
# ══════════════════════════════════════════════════════════════════

def _make_detection(bbox, candidate_type="normal", source_area=None, source_bbox=None):
    """Create a detection record. bbox is always (x, y, w, h)."""
    return {
        "bbox": tuple(int(v) for v in bbox),
        "candidate_type": candidate_type,
        "source_area": float(source_area) if source_area is not None else "",
        "source_bbox": tuple(int(v) for v in source_bbox) if source_bbox is not None else tuple(int(v) for v in bbox),
        "static_suspect": False,
    }



def _bbox_iou(a, b):
    """Return intersection-over-union for two (x, y, w, h) boxes."""
    ax, ay, aw, ah = a
    bx, by, bw, bh = b
    ax2, ay2 = ax + aw, ay + ah
    bx2, by2 = bx + bw, by + bh
    ix1, iy1 = max(ax, bx), max(ay, by)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0, ix2 - ix1), max(0, iy2 - iy1)
    inter = iw * ih
    union = aw * ah + bw * bh - inter
    return inter / union if union > 0 else 0.0


def _tile_score(image, mask, bbox):
    """Score one tile for large-motion sampling.

    The score is intentionally simple. It prefers tiles with enough foreground
    motion and visual texture, but it avoids keeping only the most foreground-
    heavy tiles because moving flowers can dominate the mask.
    """
    x, y, w, h = bbox
    tile_mask = mask[y:y+h, x:x+w]
    tile_img = image[y:y+h, x:x+w]
    if tile_mask.size == 0 or tile_img.size == 0:
        return 0.0

    fg_frac = np.count_nonzero(tile_mask) / float(w * h)
    gray = cv2.cvtColor(tile_img, cv2.COLOR_BGR2GRAY)
    texture = float(gray.std()) / 80.0
    texture = min(texture, 1.0)

    # Reward foreground up to a moderate level; very high foreground can be a
    # moving flower/background patch rather than an insect.
    fg_score = min(fg_frac / 0.12, 1.0)
    if fg_frac > 0.45:
        fg_score *= 0.75

    return 0.65 * fg_score + 0.35 * texture


def _select_tiles_with_nms(scored_tiles, max_tiles, iou_threshold):
    """Select high-scoring tiles while removing spatial duplicates.

    Returns (score, bbox) pairs so callers can keep scores for a second
    selection pass, or extract only the bbox values when needed.
    """
    selected = []
    for score, bbox in sorted(scored_tiles, key=lambda item: item[0], reverse=True):
        if all(_bbox_iou(bbox, kept_bbox) <= iou_threshold for _, kept_bbox in selected):
            selected.append((score, bbox))
        if len(selected) >= max_tiles:
            break
    return selected


def _tile_large_motion_region(image, mask, bbox, cfg):
    """Split a large moving region into a limited set of useful square tiles.

    Large motion regions often happen when a bumblebee pulls a flower or stem.
    We first generate overlapping multi-scale tiles, then score and de-duplicate
    them. This keeps high recall while avoiding a large number of near-identical
    crops from the same region.
    """
    x, y, w, h = bbox
    H, W = mask.shape[:2]

    tile_sizes = cfg.get("large_motion_tile_sizes", [320, 512])
    tile_sizes = [int(t) for t in tile_sizes]
    stride_frac = float(cfg.get("large_motion_tile_stride_frac", 0.65))
    max_per_size = int(cfg.get("large_motion_max_tiles_per_size", 6))
    max_total = int(cfg.get("large_motion_max_tiles_total", 10))
    min_fg_frac = float(cfg.get("large_motion_min_fg_frac", 0.008))
    nms_iou = float(cfg.get("large_motion_tile_nms_iou", 0.35))

    all_scored = []
    seen = set()

    for tile in tile_sizes:
        tile = min(tile, W, H)
        if tile <= 0:
            continue
        stride = max(1, int(tile * stride_frac))

        xs = list(range(x, max(x + w - tile + 1, x + 1), stride))
        ys = list(range(y, max(y + h - tile + 1, y + 1), stride))
        if not xs:
            xs = [x + w // 2 - tile // 2]
        if not ys:
            ys = [y + h // 2 - tile // 2]
        xs.append(x + w - tile)
        ys.append(y + h - tile)

        scored_for_size = []
        for yy in ys:
            for xx in xs:
                xx = max(0, min(int(xx), W - tile)) if W > tile else 0
                yy = max(0, min(int(yy), H - tile)) if H > tile else 0
                tw = min(tile, W - xx)
                th = min(tile, H - yy)
                key = (xx, yy, tw, th)
                if key in seen or tw <= 0 or th <= 0:
                    continue
                fg_frac = np.count_nonzero(mask[yy:yy+th, xx:xx+tw]) / float(tw * th)
                if fg_frac < min_fg_frac:
                    continue
                seen.add(key)
                scored_for_size.append((_tile_score(image, mask, key), key))

        all_scored.extend(
            _select_tiles_with_nms(scored_for_size, max_per_size, nms_iou)
        )

    return [bbox for _, bbox in _select_tiles_with_nms(all_scored, max_total, nms_iou)]


def _fixed_square_bbox(cx, cy, size, image_w, image_h):
    """Return a square bbox centered at (cx, cy), clipped to image bounds."""
    size = int(size)
    half = size // 2
    x1 = int(round(cx - half))
    y1 = int(round(cy - half))
    x1 = max(0, min(x1, max(0, image_w - size)))
    y1 = max(0, min(y1, max(0, image_h - size)))
    x2 = min(image_w, x1 + size)
    y2 = min(image_h, y1 + size)
    return (x1, y1, x2 - x1, y2 - y1)


def _large_motion_fallback_crops(mask, bbox, cfg):
    """Create extra crops biased toward the lower part of a large motion region.

    Large foreground regions are often caused by a pollinator pulling flowers or
    stems. The insect can be near the lower edge of the motion region, while the
    region centre is dominated by flowers. These fallback crops reduce the risk
    of missing a complete bumblebee when regular tiles only cover part of it.
    """
    x, y, w, h = bbox
    H, W = mask.shape[:2]
    sizes = [int(v) for v in cfg.get("large_motion_fallback_sizes", [512, 640])]
    centers = cfg.get("large_motion_fallback_centers", [(0.5, 0.75), (0.35, 0.75), (0.65, 0.75)])
    max_fallbacks = int(cfg.get("large_motion_max_fallbacks", 8))
    min_fg_frac = float(cfg.get("large_motion_fallback_min_fg_frac", 0.003))

    crops = []
    seen = set()
    for size in sizes:
        # Do not request a crop larger than the image, but keep it square when possible.
        size = min(size, W, H)
        if size <= 0:
            continue
        for rx, ry in centers:
            cx = x + w * float(rx)
            cy = y + h * float(ry)
            bx = _fixed_square_bbox(cx, cy, size, W, H)
            if bx in seen:
                continue
            bx0, by0, bw, bh = bx
            if bw <= 0 or bh <= 0:
                continue
            fg_frac = np.count_nonzero(mask[by0:by0+bh, bx0:bx0+bw]) / float(bw * bh)
            if fg_frac < min_fg_frac:
                continue
            seen.add(bx)
            crops.append(bx)
            if len(crops) >= max_fallbacks:
                return crops
    return crops


def detect_visitor(image, background, zone, cfg):
    """
    Find high-recall candidate insect regions in one frame by comparing to background.

    Returns a list of detection dictionaries. Candidate types:
      normal               — regular contour passing geometric filters
      large_motion_context  — full large motion region kept for manual review
      large_motion_tile     — overlapping tile from a large motion region
      large_motion_fallback — lower-region crop for insects near region edges

    Large motion is intentionally retained because true pollinator visits can
    move flowers/stems and create a large foreground component.
    """
    gray_img = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    gray_bg  = cv2.cvtColor(background, cv2.COLOR_BGR2GRAY)

    gray_img_roi = cv2.bitwise_and(gray_img, gray_img, mask=zone)
    gray_bg_roi  = cv2.bitwise_and(gray_bg,  gray_bg,  mask=zone)

    diff     = cv2.absdiff(gray_bg_roi, gray_img_roi)
    diff     = cv2.GaussianBlur(diff, (7, 7), 0)
    _, mask  = cv2.threshold(diff, cfg["darker_threshold"], 255, cv2.THRESH_BINARY)
    mask     = cv2.bitwise_and(mask, zone)

    # Exclude green vegetation. This reduces obvious grass/leaf detections, but
    # classifier-based filtering still handles remaining false positives.
    hsv   = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    green = cv2.inRange(hsv, np.array([25, 40, 40]), np.array([95, 255, 255]))
    mask  = cv2.bitwise_and(mask, cv2.bitwise_not(green))

    ko = cfg.get("kernel_open_size",  3)
    kc = cfg.get("kernel_close_size", 11)
    kernel_open  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (ko, ko))
    kernel_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kc, kc))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  kernel_open)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel_close)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    normal = []
    detections = []
    max_normal = cfg.get("max_contour_area", 35000)
    max_large = cfg.get("max_large_motion_area", 600000)

    for c in contours:
        area = cv2.contourArea(c)
        if area < cfg["min_contour_area"]:
            continue
        x, y, w, h = cv2.boundingRect(c)
        aspect = max(w, h) / max(min(w, h), 1)

        if area > max_normal:
            if area <= max_large:
                source_bbox = (x, y, w, h)
                # Keep the full region as context for manual inspection.
                detections.append(_make_detection(source_bbox, "large_motion_context", area, source_bbox))
                # Also tile the region so the insect is not missed when the
                # contour centre is the flower/background instead of the insect.
                for tb in _tile_large_motion_region(image, mask, source_bbox, cfg):
                    detections.append(_make_detection(tb, "large_motion_tile", area, source_bbox))
                # Add lower-region fallback crops. These are important when the
                # flower dominates the motion region but the insect hangs below it.
                for fb in _large_motion_fallback_crops(mask, source_bbox, cfg):
                    detections.append(_make_detection(fb, "large_motion_fallback", area, source_bbox))
            # If it is even larger than max_large, it is likely global lighting
            # shift or camera movement; do not save hundreds of useless crops.
            continue

        if aspect > cfg["max_aspect_ratio"]:
            continue

        normal.append((area, (x, y, w, h)))

    normal_bboxes = [bbox for _, bbox in sorted(normal, key=lambda v: v[0], reverse=True)]
    normal_bboxes = merge_nearby_bboxes(
        normal_bboxes,
        merge_dist=cfg.get("merge_dist", 20),
        max_merged_area=cfg.get("max_contour_area", 35000),
    )

    min_px = cfg.get("min_crop_px", 10)
    for x, y, w, h in normal_bboxes:
        if max(w, h) >= min_px:
            detections.append(_make_detection((x, y, w, h), "normal", w*h, (x, y, w, h)))

    return detections

def merge_nearby_bboxes(bboxes, merge_dist=20, max_merged_area=35000):
    """
    Merge bounding boxes that overlap, contain each other, or are very close.

    Strategy (applied repeatedly until stable):
    1. Overlap / containment — boxes that intersect are always merged
    2. Near gap — boxes within merge_dist pixels of each other are merged

    If the merged result would exceed max_merged_area, the merge is rejected
    and the original small boxes are kept instead. This handles cases where
    a large moving object (e.g. flower pulled down by a bumblebee) creates
    a huge contour — the individual insect body parts are preserved rather
    than being swallowed into one giant box that gets filtered out.

    merge_dist=20px is conservative: fly body parts split by background
    subtraction are usually <20px apart, while two separate insects are
    usually further apart.
    """
    if not bboxes:
        return []

    boxes = list(bboxes)
    changed = True
    while changed:
        changed = False
        merged = []
        used = set()

        for i, (x1, y1, w1, h1) in enumerate(boxes):
            if i in used:
                continue
            gx1, gy1, gx2, gy2 = x1, y1, x1+w1, y1+h1
            group = [i]

            for j, (x2, y2, w2, h2) in enumerate(boxes):
                if j <= i or j in used:
                    continue
                bx1, by1, bx2, by2 = x2, y2, x2+w2, y2+h2

                # Gap between boxes (0 if overlapping)
                gap_x = max(0, max(bx1, gx1) - min(bx2, gx2))
                gap_y = max(0, max(by1, gy1) - min(by2, gy2))

                if gap_x <= merge_dist and gap_y <= merge_dist:
                    # Check if merging would exceed max size
                    new_gx1 = min(gx1, bx1)
                    new_gy1 = min(gy1, by1)
                    new_gx2 = max(gx2, bx2)
                    new_gy2 = max(gy2, by2)
                    merged_area = (new_gx2 - new_gx1) * (new_gy2 - new_gy1)

                    if merged_area <= max_merged_area:
                        # Safe to merge
                        gx1, gy1, gx2, gy2 = new_gx1, new_gy1, new_gx2, new_gy2
                        used.add(j)
                        group.append(j)
                        changed = True
                    # else: skip this merge — keep both boxes separately

            used.add(i)
            merged.append((gx1, gy1, gx2-gx1, gy2-gy1))

        boxes = merged

    return boxes


def detect_all_frames(paths, background, zone, cfg, debug=False, debug_dir=None, image_dir=None):
    """
    Step 4: Run background subtraction on every frame.

    If cfg['rolling_window'] > 0, uses a rolling median background instead of
    the global median, further reducing slow lighting-change false positives.
    """
    print("Detecting visitors...")
    all_detections = {}
    _strip_temps    = {}   # path -> temperature from OSD strip (when OCR enabled)
    window       = cfg.get("rolling_window", 0)
    frames_cache = []  # for rolling background

    # Gap-reset: when EXIF time between successive frames exceeds this many
    # seconds, drop the rolling cache so the next frame seeds a fresh bg
    # (otherwise it gets diffed against a frame from minutes/hours earlier
    # and produces nonsense detections after triage frame jumps).
    # Auto-enable gap-reset (90s) when running from a flag list — without
    # it the first frame of each sparse cluster gets diffed against a frame
    # from a different time of day and emits garbage detections.
    gap_reset = 90 if cfg.get("frames_from") else 0
    _prev_ts  = None
    _exif_dt_tag = next((k for k, v in TAGS.items() if v == "DateTimeOriginal"), None)

    def _read_ts(p):
        if _exif_dt_tag is None:
            return None
        try:
            img = PILImage.open(p)
            exif = img._getexif() if hasattr(img, "_getexif") else dict(img.getexif())
            s = exif.get(_exif_dt_tag) if exif else None
            if not s:
                return None
            date, t = s.split(" ")
            y, m, d = map(int, date.split(":"))
            hh, mm, ss = map(int, t.split(":"))
            import datetime as _dt
            return _dt.datetime(y, m, d, hh, mm, ss).timestamp()
        except Exception:
            return None

    import time as _time
    progress_every = int(cfg.get("progress_every", 50))
    t_start = _time.time()

    total = len(paths)
    for i, path in enumerate(paths):
        image = cv2.imread(str(path))
        if image is None:
            continue

        # Reset rolling cache after a temporal gap (e.g. triage frame jumps)
        if gap_reset > 0:
            ts = _read_ts(path)
            if ts is not None and _prev_ts is not None and (ts - _prev_ts) > gap_reset:
                frames_cache.clear()
            if ts is not None:
                _prev_ts = ts

        # Strip bottom info bar and extract temperature (if OCR is enabled)
        image, strip_temp = preprocess_image(image, cfg)
        if strip_temp is not None:
            _strip_temps[str(path)] = strip_temp
        # Crop zone to match stripped image height
        if image is not None and zone is not None and zone.shape[0] != image.shape[0]:
            zone = zone[:image.shape[0], :image.shape[1]]

        # Choose background: rolling (last N frames) or global median
        if window > 0 and len(frames_cache) >= 1:
            # Rolling background: grass that moves consistently across recent
            # frames is absorbed into the background and will not be detected.
            if window == 1:
                # Median of a single frame is the frame itself — skip np.median
                # (which copies the whole array) and reference it directly.
                bg = frames_cache[-1]
            else:
                bg = np.median(frames_cache[-window:], axis=0).astype(np.uint8)
        else:
            # First frame (rolling) or global mode: use precomputed global background
            bg = background

        # No background yet (first frame in rolling-only mode with no global bg):
        # record an empty detection list, seed the cache, move on.
        if bg is None:
            all_detections[str(path)] = []
            frames_cache.append(image)
            if window > 0 and len(frames_cache) > window:
                frames_cache.pop(0)
            continue

        img_to_detect = image

        if debug and debug_dir and _debug_intermediate_enabled(cfg):
            # Build debug name with cam_prefix to match crop filename convention
            if image_dir:
                _idir = Path(image_dir)
                _dbg_prefix = f"{_idir.parent.name}_{_idir.name}"
                _dbg_stem   = str(Path(path).relative_to(image_dir)).replace("/", "_").replace("\\", "_").rsplit(".", 1)[0]
                _dbg_name   = f"{_dbg_prefix}__{_dbg_stem}"
            else:
                _dbg_name = path.stem
            # Crop bg to match current image size (bg may be full-size if strip was removed)
            _bg_dbg = bg[:image.shape[0], :image.shape[1]] if bg.shape[:2] != image.shape[:2] else bg
            save_debug_images(_dbg_name, image, _bg_dbg, zone, None, debug_dir, cfg)

        # Ensure bg matches current image size before detection
        _bg_det = bg[:img_to_detect.shape[0], :img_to_detect.shape[1]] if bg.shape[:2] != img_to_detect.shape[:2] else bg
        all_detections[str(path)] = detect_visitor(img_to_detect, _bg_det, zone, cfg)

        # Update histories
        frames_cache.append(image)
        if window > 0 and len(frames_cache) > window:
            frames_cache.pop(0)

        if progress_every > 0 and (i + 1) % progress_every == 0:
            elapsed = _time.time() - t_start
            fps = (i + 1) / elapsed if elapsed > 0 else 0
            eta = (total - (i + 1)) / fps if fps > 0 else 0
            print(f"  [{i+1}/{total}] {fps:.1f} fps, ETA {eta:5.0f}s", flush=True)

    elapsed = _time.time() - t_start
    fps = total / elapsed if elapsed > 0 else 0
    print(f"  done — {total} frames in {elapsed:.1f}s ({fps:.1f} fps)")
    return all_detections, _strip_temps


def _det_bbox(det):
    """Return bbox from either legacy tuple detection or new detection dict."""
    if isinstance(det, dict):
        return tuple(det["bbox"])
    return tuple(det)


def _with_static_flag(det, flag):
    """Return a detection dict with static_suspect set."""
    if isinstance(det, dict):
        out = dict(det)
    else:
        out = _make_detection(det, "normal")
    out["static_suspect"] = bool(flag)
    return out


def filter_static_detections(all_detections, cfg, paths=None):
    """
    Mark or remove detections that appear at the same position for too many
    consecutive frames.

    In high-recall mode (static_filter_mode='flag'), static suspects are kept
    and written to crops/CSV with static_suspect=True. This avoids losing real
    insects that pass the contour stage. Set static_filter_mode='drop' to use
    the old hard-removal behavior.
    """
    dist  = cfg["static_dist"]
    max_f = cfg["static_max_frames"]
    mode  = cfg.get("static_filter_mode", "flag")

    if not paths:
        paths = list(all_detections.keys())

    frame_centers = {}
    for p in paths:
        dets = all_detections.get(str(p), all_detections.get(p, []))
        centers = []
        for det in dets:
            x, y, w, h = _det_bbox(det)
            centers.append((x + w//2, y + h//2))
        frame_centers[str(p)] = centers

    static_positions = set()
    n = len(paths)
    for i, p in enumerate(paths):
        for cx, cy in frame_centers.get(str(p), []):
            consecutive = 0
            for j in range(i, n):
                pj = str(paths[j])
                has_nearby = any(
                    ((cx - cx2)**2 + (cy - cy2)**2)**0.5 < dist
                    for cx2, cy2 in frame_centers.get(pj, [])
                )
                if has_nearby:
                    consecutive += 1
                else:
                    break
            if consecutive > max_f:
                static_positions.add((cx, cy))

    if static_positions:
        action = "Flagged" if mode != "drop" else "Suppressed"
        print(f"{action} {len(static_positions)} static position(s) "
              f"(>{max_f} consecutive frames)")

    filtered = {}
    for p in paths:
        p_str = str(p)
        dets = all_detections.get(p_str, all_detections.get(p, []))
        out = []
        for det in dets:
            x, y, w, h = _det_bbox(det)
            is_static = any(
                ((x + w//2 - sx)**2 + (y + h//2 - sy)**2)**0.5 < dist
                for sx, sy in static_positions
            )
            if mode == "drop" and is_static:
                continue
            out.append(_with_static_flag(det, is_static))
        filtered[p_str] = out
    return filtered


def crop_with_padding(image, detection, cfg):
    """Return crop for a detection.

    Normal candidates use an adaptive fixed-size square crop centered on the
    bbox. Large-motion context crops keep the full large region. Large-motion
    tiles and fallback crops are saved exactly as generated. This avoids the
    problem where a single center crop of a large flower-motion bbox misses the
    bumblebee.
    """
    det = detection if isinstance(detection, dict) else _make_detection(detection, "normal")
    x, y, w, h = _det_bbox(det)
    h_img, w_img = image.shape[:2]
    candidate_type = det.get("candidate_type", "normal")

    if candidate_type == "large_motion_context":
        pad = int(cfg.get("large_motion_context_pad", 10))
        x1 = max(0, x - pad)
        y1 = max(0, y - pad)
        x2 = min(w_img, x + w + pad)
        y2 = min(h_img, y + h + pad)
        return image[y1:y2, x1:x2]

    if candidate_type in {"large_motion_tile", "large_motion_fallback"}:
        return image[y:y+h, x:x+w]

    # Normal adaptive crop for small/medium candidates.
    eff = (w * h) ** 0.5
    if eff < 50:
        window = int(eff * 2.5)
        window = max(50, min(window, 180))
    elif eff < 120:
        window = int(eff * 1.8)
        window = max(80, min(window, 260))
    else:
        window = int(eff * 1.2)
        window = max(140, min(window, 320))

    pad_ratio = 0.15   # 15% padding around the bbox, clipped to image bounds
    pad_x = int(w * pad_ratio)
    pad_y = int(h * pad_ratio)
    x1 = max(0, x - pad_x)
    y1 = max(0, y - pad_y)
    x2 = min(w_img, x + w + pad_x)
    y2 = min(h_img, y + h + pad_y)
    return image[y1:y2, x1:x2]


def _debug_output_set(cfg):
    """Normalize cfg['debug_outputs'] into a set of output kinds."""
    outputs = cfg.get("debug_outputs", "all")
    if outputs is True:
        outputs = "all"
    if outputs in (False, None):
        return set()
    if isinstance(outputs, str):
        value = outputs.strip().lower()
        if value in {"all", "full"}:
            return {"original", "diff", "contours", "final"}
        if value in {"annotate", "annotation", "annotate_only"}:
            return {"final"}
        if value in {"none", "off", "false"}:
            return set()
        return {part.strip().lower() for part in value.split(",") if part.strip()}
    return {str(part).strip().lower() for part in outputs}


def _debug_output_enabled(cfg, kind):
    return kind in _debug_output_set(cfg)


def _debug_intermediate_enabled(cfg):
    return bool(_debug_output_set(cfg) & {"original", "diff", "contours"})


def _resize_debug_for_storage(img, cfg):
    """Downscale debug images only. Detection crops are never resized here."""
    h, w = img.shape[:2]
    scale = 1.0
    max_w = cfg.get("debug_max_width") or 0
    max_h = cfg.get("debug_max_height") or 0
    if max_w and w > max_w:
        scale = min(scale, max_w / w)
    if max_h and h > max_h:
        scale = min(scale, max_h / h)
    manual_scale = cfg.get("debug_scale")
    if manual_scale and 0 < manual_scale < 1:
        scale = min(scale, float(manual_scale))
    if scale >= 1.0:
        return img, 1.0
    new_w = max(1, int(round(w * scale)))
    new_h = max(1, int(round(h * scale)))
    return cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA), scale


def _write_debug_image(path, img, cfg):
    img_out, storage_scale = _resize_debug_for_storage(img, cfg)
    quality = int(cfg.get("debug_jpeg_quality", 90))
    quality = max(1, min(100, quality))
    params = [cv2.IMWRITE_JPEG_QUALITY, quality]
    cv2.imwrite(str(path), img_out, params)
    return storage_scale


def save_final_debug_image(name, image, detections, debug_dir, zone, cfg):
    """Save overlay of final saved crops, cropped to ROI region when ROI is active."""
    import json as _json
    if not _debug_output_enabled(cfg, "final"):
        return
    if not detections and not cfg.get("debug_save_empty_frames", True):
        return
    overlay = image.copy()

    # Compute ROI crop bounds (used for drawing + final crop)
    _crop_x1 = _crop_y1 = 0
    _crop_x2, _crop_y2 = overlay.shape[1], overlay.shape[0]
    _do_roi_crop = False

    zone_coords = cv2.findNonZero(zone)
    if zone_coords is not None:
        zx, zy, zw, zh = cv2.boundingRect(zone_coords)
        zone_coverage = np.count_nonzero(zone) / max(1, zone.size)
        if zone_coverage < 0.95:
            cv2.rectangle(overlay, (zx, zy), (zx+zw, zy+zh), (0, 165, 255), 4)
            cv2.putText(overlay, "ROI", (zx+4, zy+20),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 165, 255), 2)
            # Pad 40 px around ROI
            pad = 40
            ih, iw = overlay.shape[:2]
            _crop_x1 = max(0, zx - pad)
            _crop_y1 = max(0, zy - pad)
            _crop_x2 = min(iw, zx + zw + pad)
            _crop_y2 = min(ih, zy + zh + pad)
            _do_roi_crop = True

    for det in detections:
        x, y, w, h = _det_bbox(det)
        ctype = det.get("candidate_type", "normal") if isinstance(det, dict) else "normal"
        static = det.get("static_suspect", False) if isinstance(det, dict) else False
        in_roi_d = is_near_marked_flower((x, y, w, h), zone, cfg)
        if static and in_roi_d:
            color = (0, 255, 255)
        elif static:
            color = (180, 105, 255)
        elif in_roi_d:
            color = (0, 255, 0)
        else:
            color = (255, 0, 0)
        cv2.rectangle(overlay, (x, y), (x+w, y+h), color, 2)
        label_txt = ("IN " if in_roi_d else "OUT ") + ctype.replace("large_motion_", "lm_")
        cv2.putText(overlay, label_txt, (x, y-8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.40, color, 1)

    # Crop to ROI region (after drawing so boxes appear in cropped image)
    roi_ox, roi_oy = 0, 0
    if _do_roi_crop:
        overlay = overlay[_crop_y1:_crop_y2, _crop_x1:_crop_x2]
        roi_ox, roi_oy = _crop_x1, _crop_y1

    storage_scale = _write_debug_image(debug_dir / f"{name}_4_final_saved_crops.jpg", overlay, cfg)

    # Save ROI offset so annotate.py can map original-image bbox coords onto cropped debug image
    offset_path = debug_dir / f"{name}_4_offset.json"
    offset_path.write_text(_json.dumps({"ox": roi_ox, "oy": roi_oy, "scale": storage_scale}))

# ══════════════════════════════════════════════════════════════════
# WEATHER (EXIF)
# ══════════════════════════════════════════════════════════════════


# ══════════════════════════════════════════════════════════════════
# BOTTOM INFO STRIP — crop out camera OSD and extract temperature
# ══════════════════════════════════════════════════════════════════

def extract_strip_temperature(strip_bgr):
    """OCR the temperature value from the camera info strip.

    Looks for patterns like: 19C  -3C  19  23  -3.5
    Tries both white-on-dark and dark-on-white thresholding.
    Returns temperature as float, or None if not found.
    """
    try:
        import re
        import pytesseract
        from PIL import Image as _PILI

        h, w = strip_bgr.shape[:2]
        # Always upscale to at least 120px height for reliable OCR
        target_h = 120
        scale = max(2, target_h // max(h, 1))
        strip_up = cv2.resize(strip_bgr, (w * scale, h * scale),
                              interpolation=cv2.INTER_CUBIC)
        gray = cv2.cvtColor(strip_up, cv2.COLOR_BGR2GRAY)

        # Try both thresholding directions (camera may be white or dark text)
        best_txt = ""
        for thresh_type in [cv2.THRESH_BINARY, cv2.THRESH_BINARY_INV]:
            _, bw = cv2.threshold(gray, 0, 255,
                                  thresh_type | cv2.THRESH_OTSU)
            txt = pytesseract.image_to_string(
                _PILI.fromarray(bw),
                config="--psm 6 -c tessedit_char_whitelist=0123456789-. "
            )
            if len(txt.strip()) > len(best_txt.strip()):
                best_txt = txt

        # Find any number between -50 and 60 — temperature candidate
        # Camera format is usually just "19" or "-3" near the start of the strip
        numbers = re.findall(r"-?\d{1,3}(?:\.\d)?", best_txt)
        for n in numbers:
            val = float(n)
            if -50 <= val <= 60:
                return val
    except Exception:
        pass
    return None


def preprocess_image(image_bgr, cfg):
    """Crop the bottom info strip and optionally OCR the temperature.

    Returns:
        processed_image : image with strip removed (or original if strip_height=0)
        temperature     : float or None (None when OCR is disabled)
    """
    strip_h = int(cfg.get("strip_height", 0))
    if strip_h <= 0 or image_bgr is None:
        return image_bgr, None

    h = image_bgr.shape[0]
    if strip_h >= h:
        return image_bgr, None

    temp = None
    if cfg.get("strip_ocr_temperature", False):
        strip = image_bgr[h - strip_h:, :]
        temp  = extract_strip_temperature(strip)

    return image_bgr[:h - strip_h, :], temp

def get_exif_metadata(image_path, cfg):
    """
    Step 3: Extract metadata, classify weather, and detect flash/fog.

    Camera: Wingscapes TLCAM PRO — fixed aperture f/2.8, auto exposure.
    Shutter speed reflects ambient light:
      sunny  → fast shutter → denominator > sunny_shutter_threshold
      cloudy → slow shutter → denominator <= sunny_shutter_threshold

    Flash detection: EXIF tag 37385 (Flash). Nonzero = flash fired → night/indoor shot.
    Foggy detection: Laplacian variance of grayscale image. Low variance = low contrast → fog/blur.

    Returns metadata dict with a "skip" key and "skip_reason" if the image should be excluded.
    """
    metadata = {
        "datetime": "", "camera_name": "", "shutter_speed": "",
        "weather": "unknown", "flash": False,
        "laplacian_var": -1.0,
        "skip": False, "skip_reason": "",
    }
    try:
        img_pil   = PILImage.open(image_path)
        exif_data = img_pil._getexif() if hasattr(img_pil, "_getexif") else dict(img_pil.getexif())  # type: ignore[attr-defined]
        if exif_data is None:
            return metadata
        tag_map = {TAGS.get(k, k): v for k, v in exif_data.items()}
        metadata["datetime"]    = str(tag_map.get("DateTimeOriginal", ""))
        metadata["camera_name"] = str(tag_map.get("Model", ""))

        # Weather from shutter speed
        exposure = exif_data.get(33434)  # ExposureTime
        if exposure:
            denom = exposure[1] if isinstance(exposure, tuple) else int(1 / exposure)
            metadata["shutter_speed"] = f"1/{denom}"
            metadata["weather"] = "sunny" if denom > cfg["sunny_shutter_threshold"] else "cloudy"

        # Flash detection — EXIF tag 37385
        # Value 0x0 = no flash, any nonzero value = flash fired (various modes)
        flash_val = exif_data.get(37385, 0)
        if flash_val and int(flash_val) != 0:
            metadata["flash"] = True
            if cfg.get("skip_flash", True):
                metadata["skip"]        = True
                metadata["skip_reason"] = f"flash (EXIF={flash_val})"

    except Exception:
        pass

    # Foggy/blurry detection — Laplacian variance on grayscale
    # Low variance = low contrast = fog, heavy cloud, or blur
    try:
        img = cv2.imread(str(image_path))
        if img is not None:
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            lap_var = cv2.Laplacian(gray, cv2.CV_64F).var()
            metadata["laplacian_var"] = round(float(lap_var), 1)
            if not metadata["skip"] and cfg.get("skip_foggy", True):
                if lap_var < cfg.get("foggy_threshold", 50):
                    metadata["skip"]        = True
                    metadata["skip_reason"] = f"foggy/blurry (laplacian={lap_var:.1f})"
    except Exception:
        pass

    return metadata


# ══════════════════════════════════════════════════════════════════
# CLASSIFICATION HELPERS
# ══════════════════════════════════════════════════════════════════

def is_near_marked_flower(bbox, zone, cfg):
    """
    Returns True if the insect bounding box overlaps sufficiently with the ROI zone.
    Uses near_flower_iou_threshold as the minimum overlap fraction.
    """
    x, y, w, h = bbox
    roi_crop = zone[y:y+h, x:x+w]
    if roi_crop.size == 0:
        return False
    return np.count_nonzero(roi_crop) / roi_crop.size > cfg["near_flower_iou_threshold"]


# ══════════════════════════════════════════════════════════════════
# CSV
# ══════════════════════════════════════════════════════════════════

def init_csv(output_path, fields):
    """Create CSV file with headers. Overwrites existing file."""
    with open(output_path, "w", newline="") as f:
        csv.DictWriter(f, fieldnames=fields).writeheader()


def write_csv_row(output_path, row_dict, fields):
    """Append one row to the CSV. Missing fields are written as empty string."""
    with open(output_path, "a", newline="") as f:
        csv.DictWriter(f, fieldnames=fields).writerow(
            {k: row_dict.get(k, "") for k in fields}
        )


# ══════════════════════════════════════════════════════════════════
# DEBUG IMAGES
# ══════════════════════════════════════════════════════════════════

def save_debug_images(name, image, background, zone, marker, debug_dir, cfg, detections=None):
    """Save debug images cropped to ROI bounding box for easy inspection.

    _3_contours color legend:
      green  = normal contour that passes filters
      orange = large motion region retained for context/tiling/fallback crops
      red    = rejected contour
    _4_final_saved_crops is written later after static flagging.
    """
    coords = cv2.findNonZero(zone)
    if coords is not None:
        rx, ry, rw, rh = cv2.boundingRect(coords)
    else:
        rx, ry = 0, 0
        rh, rw = image.shape[:2]

    def roi_crop(img):
        return img[ry:ry+rh, rx:rx+rw]

    outputs = _debug_output_set(cfg)
    if "original" in outputs:
        _write_debug_image(debug_dir / f"{name}_1_original.jpg", image, cfg)
    if not (outputs & {"diff", "contours"}):
        return

    gray_img = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    gray_bg  = cv2.cvtColor(background, cv2.COLOR_BGR2GRAY)
    gray_img = cv2.bitwise_and(gray_img, gray_img, mask=zone)
    gray_bg  = cv2.bitwise_and(gray_bg,  gray_bg,  mask=zone)
    diff     = cv2.absdiff(gray_bg, gray_img)
    diff     = cv2.GaussianBlur(diff, (7, 7), 0)
    _, mask  = cv2.threshold(diff, cfg["darker_threshold"], 255, cv2.THRESH_BINARY)
    mask     = cv2.bitwise_and(mask, zone)
    hsv      = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    green    = cv2.inRange(hsv, np.array([25,40,40]), np.array([95,255,255]))
    mask     = cv2.bitwise_and(mask, cv2.bitwise_not(green))

    ko = cfg.get("kernel_open_size",  3)
    kc = cfg.get("kernel_close_size", 11)
    kernel_open  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (ko, ko))
    kernel_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kc, kc))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  kernel_open)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel_close)
    if "diff" in outputs:
        _write_debug_image(debug_dir / f"{name}_2_diff.jpg", roi_crop(mask), cfg)
    if "contours" not in outputs:
        return

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    overlay2 = image.copy()  # always full image for contours

    # Draw ROI zone boundary (cyan) so it's clear which region is the focal plant
    zone_coords = cv2.findNonZero(zone)
    if zone_coords is not None:
        zx, zy, zw, zh = cv2.boundingRect(zone_coords)
        zone_coverage = np.count_nonzero(zone) / max(1, zone.size)
        if zone_coverage < 0.95:  # only draw if it's a real ROI, not full image
            cv2.rectangle(overlay2, (zx, zy), (zx+zw, zy+zh), (0, 165, 255), 4)  # orange ROI boundary
            cv2.putText(overlay2, "ROI | green=in | blue=out | red=filtered",
                        (zx+4, zy+20), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 165, 255), 2)
    max_normal = cfg.get("max_contour_area", 35000)
    max_large = cfg.get("max_large_motion_area", 600000)
    for c in contours:
        area = cv2.contourArea(c)
        if area < 100:
            continue
        x, y, w, h = cv2.boundingRect(c)
        aspect = max(w,h)/max(min(w,h),1)
        passes = (area >= cfg["min_contour_area"] and area <= max_normal
                  and aspect <= cfg["max_aspect_ratio"])
        large  = (area > max_normal and area <= max_large)
        if passes:
            in_roi_c = is_near_marked_flower((x, y, w, h), zone, cfg)
            color = (0, 255, 0) if in_roi_c else (255, 0, 0)   # green=in ROI, blue=out
            label = f"{'IN' if in_roi_c else 'OUT'} {w}x{h}px a={int(area)}"
        elif large:
            color = (0, 165, 255)   # orange = large motion
            label = f"large {w}x{h}px a={int(area)}"
        else:
            color = (0, 0, 255)     # red = rejected
            label = f"reject {w}x{h}px a={int(area)}"
        cv2.rectangle(overlay2, (x, y), (x+w, y+h), color, 2)
        cv2.putText(overlay2, label, (x, y-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
    _write_debug_image(debug_dir / f"{name}_3_contours.jpg", overlay2, cfg)


### Pipeline functions & Main

In [6]:
def _emit_frame_results(idx_p, total_paths, rel_id, path_str, image,
                        detections, strip_temps,
                        cam_prefix, camera_path, zone,
                        output_csv, cfg, csv_fields, crop_dir,
                        model,
                        debug, debug_dir):
    """Per-frame "save crops + debug + CSV" work shared by the two-pass
    classify_and_write and the single-pass streaming loop. `image` may be
    None for frames where the JPG decode was skipped (no detections, no
    empty-frame debug); CSV-only branches still run."""
    if image is not None and debug and debug_dir:
        _dbg_name = f"{cam_prefix}__{rel_id.rsplit('.', 1)[0]}"
        save_final_debug_image(_dbg_name, image, detections, debug_dir, zone, cfg)

    metadata = get_exif_metadata(path_str, cfg)
    metadata["camera_path"] = camera_path
    metadata["temperature_c"] = (strip_temps or {}).get(path_str, "")

    if metadata.get("skip"):
        print(f"  [{idx_p+1}/{total_paths}] {rel_id} | SKIP: {metadata['skip_reason']} | sharpness={metadata.get('laplacian_var',-1):.0f}")
        write_csv_row(output_csv, {
            "image_name":    rel_id,
            "datetime":      metadata["datetime"],
            "camera_name":   metadata["camera_name"],
            "shutter_speed": metadata["shutter_speed"],
            "weather":       metadata["weather"],
            "skip":          True,
            "skip_reason":   metadata["skip_reason"],
            "laplacian_var": metadata.get("laplacian_var", ""),
            "pollinator_detected": "skipped",
        }, csv_fields)
        return

    if not detections:
        print(f"  [{idx_p+1}/{total_paths}] {rel_id} | {metadata.get('weather','?')} | sharpness={metadata.get('laplacian_var',-1):.0f} | no detection")
        write_csv_row(output_csv,
            {"image_name": rel_id, **metadata, "pollinator_detected": "no"},
            csv_fields)
        return

    if image is None:
        # Detections exist but the image wasn't loaded — can't crop. Skip.
        return

    print(f"  [{idx_p+1}/{total_paths}] {rel_id} | {metadata.get('weather','?')} | sharpness={metadata.get('laplacian_var',-1):.0f} | {len(detections)} candidate(s)")
    for i, det in enumerate(detections):
        bbox = _det_bbox(det)
        x, y, w, h = bbox
        candidate_type = det.get("candidate_type", "normal") if isinstance(det, dict) else "normal"
        static_suspect = det.get("static_suspect", False) if isinstance(det, dict) else False
        source_area   = det.get("source_area", "") if isinstance(det, dict) else ""

        crop     = crop_with_padding(image, det, cfg)
        crop_rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)

        safe_type   = candidate_type.replace("/", "_")
        in_roi_flag = is_near_marked_flower(bbox, zone, cfg)
        scope_tag   = "roi" if in_roi_flag else "out"

        crop_mode = cfg.get("crop_mode", "all")
        if crop_mode == "roi_only" and not in_roi_flag:
            continue
        if crop_mode == "out_only" and in_roi_flag:
            continue

        crop_filename = f"{cam_prefix}__{rel_id.rsplit('.', 1)[0]}_crop_{i}_{safe_type}_{scope_tag}.jpg"

        # ── Two-stage classification ──────────────────────────────────
        binary_model, binary_tf, group_model, group_tf, group_classes, clf_device = model
        clf_result = classify_crop(
            crop, binary_model, binary_tf, group_model, group_tf,
            group_classes, clf_device,
            binary_threshold=cfg.get("binary_threshold", 0.5)
        )

        in_roi = is_near_marked_flower(bbox, zone, cfg)
        scope  = "roi" if in_roi else "outside_roi"

        is_insect = clf_result["binary_label"] == "insect"

        # Skip background crops entirely — no crop saved, no CSV row
        if not is_insect:
            continue

        # Save crop
        saved = cv2.imwrite(str(crop_dir / crop_filename), crop)
        if not saved:
            print(f"  WARNING: failed to save crop {crop_filename}")

        print(f"  {rel_id} [{scope}]: binary={clf_result['binary_label']} "
              f"({clf_result['binary_confidence']:.2f})"
              + (f" | {clf_result['pollinator_type']} ({clf_result['group_confidence']:.2f})"
                 if is_insect and clf_result['pollinator_type'] else ""))

        write_csv_row(output_csv, {
            "image_name": rel_id, **metadata,
            "pollinator_detected": "yes",
            "crop_filename":       crop_filename,
            "candidate_type":      candidate_type,
            "static_suspect":      str(static_suspect),
            "source_area":         source_area,
            "near_marked_flower":  str(in_roi),
            "detection_scope":     scope,
            "binary_label":        clf_result["binary_label"],
            "binary_confidence":   clf_result["binary_confidence"],
            "pollinator_type":     clf_result["pollinator_type"],
            "group_confidence":    clf_result["group_confidence"],
            "bumblebee_prob":      clf_result["bumblebee_prob"],
            "fly_prob":            clf_result["fly_prob"],
            "butterfly_moth_prob": clf_result["butterfly_moth_prob"],
            "other_prob":          clf_result["other_prob"],
            "bbox_x": x, "bbox_y": y, "bbox_w": w, "bbox_h": h,
        }, csv_fields)


def _camera_prefix_for(image_dir):
    if image_dir:
        _idir = Path(image_dir)
        cam_prefix = f"{_idir.parent.name}_{_idir.name}"
    else:
        cam_prefix = "cam"
    return cam_prefix, cam_prefix


def _rel_id_for(path, image_dir):
    if image_dir:
        try:
            return str(Path(path).relative_to(image_dir)).replace("/", "_").replace("\\", "_")
        except ValueError:
            return Path(path).name
    return Path(path).name


def detect_and_classify_streaming(paths, zone, model,
                                  output_csv, cfg, csv_fields, crop_dir,
                                  background=None, debug=False, debug_dir=None,
                                  image_dir=None):
    """
    Single-pass: detect + crop + CSV in one loop. Used when the static filter
    is disabled, so there's no need to collect every detection before saving.
    Halves the per-frame I/O compared to detect_all_frames + classify_and_write.
    """
    init_csv(output_csv, csv_fields)
    cam_prefix, camera_path = _camera_prefix_for(image_dir)
    print("Detecting + classifying visitors (single-pass)...")

    window       = cfg.get("rolling_window", 0)
    frames_cache = []
    strip_temps  = {}

    # Gap-reset: same logic as detect_all_frames — clear the rolling cache
    # when EXIF time between consecutive processed frames exceeds the
    # threshold (e.g. flagged-frame jumps). The next frame seeds a fresh
    # bg and produces no detections.
    # Auto-enable gap-reset (90s) when running from a flag list — without
    # it the first frame of each sparse cluster gets diffed against a frame
    # from a different time of day and emits garbage detections.
    gap_reset = 90 if cfg.get("frames_from") else 0
    _prev_ts  = None
    _exif_dt_tag = next((k for k, v in TAGS.items() if v == "DateTimeOriginal"), None)

    def _read_ts(p):
        if _exif_dt_tag is None:
            return None
        try:
            img = PILImage.open(p)
            exif = img._getexif() if hasattr(img, "_getexif") else dict(img.getexif())
            s = exif.get(_exif_dt_tag) if exif else None
            if not s:
                return None
            date, t = s.split(" ")
            y, m, d = map(int, date.split(":"))
            hh, mm, ss = map(int, t.split(":"))
            import datetime as _dt
            return _dt.datetime(y, m, d, hh, mm, ss).timestamp()
        except Exception:
            return None

    import time as _time
    progress_every = int(cfg.get("progress_every", 50))
    t_start = _time.time()
    total = len(paths)

    for idx_p, path in enumerate(paths):
        path_str = str(path)
        rel_id   = _rel_id_for(path, image_dir)

        image = cv2.imread(path_str)
        if image is None:
            continue

        if gap_reset > 0:
            ts = _read_ts(path)
            if ts is not None and _prev_ts is not None and (ts - _prev_ts) > gap_reset:
                frames_cache.clear()
            if ts is not None:
                _prev_ts = ts

        image, strip_temp = preprocess_image(image, cfg)
        if strip_temp is not None:
            strip_temps[path_str] = strip_temp

        if zone is not None and zone.shape[0] != image.shape[0]:
            zone = zone[:image.shape[0], :image.shape[1]]

        if window > 0 and len(frames_cache) >= 1:
            if window == 1:
                bg = frames_cache[-1]
            else:
                bg = np.median(frames_cache[-window:], axis=0).astype(np.uint8)
        else:
            bg = background

        if bg is None:
            # First frame in rolling-only mode: no detection possible.
            detections = []
        else:
            _bg_det = bg[:image.shape[0], :image.shape[1]] if bg.shape[:2] != image.shape[:2] else bg
            detections = detect_visitor(image, _bg_det, zone, cfg)

        frames_cache.append(image)
        if window > 0 and len(frames_cache) > window:
            frames_cache.pop(0)

        _emit_frame_results(
            idx_p, total, rel_id, path_str, image,
            detections, strip_temps,
            cam_prefix, camera_path, zone,
            output_csv, cfg, csv_fields, crop_dir,
            model,
            debug, debug_dir,
        )

        if progress_every > 0 and (idx_p + 1) % progress_every == 0:
            elapsed = _time.time() - t_start
            fps = (idx_p + 1) / elapsed if elapsed > 0 else 0
            eta = (total - (idx_p + 1)) / fps if fps > 0 else 0
            print(f"  [{idx_p+1}/{total}] {fps:.1f} fps, ETA {eta:5.0f}s", flush=True)

    elapsed = _time.time() - t_start
    fps = total / elapsed if elapsed > 0 else 0
    print(f"  done — {total} frames in {elapsed:.1f}s ({fps:.1f} fps)")


def classify_and_write(paths, filtered_full, zone, model,
                       output_csv, cfg, csv_fields,
                       crop_dir, debug=False, debug_dir=None, image_dir=None,
                       strip_temps=None):
    """
    Two-pass classifier: re-reads each frame, crops surviving detections,
    writes CSV rows + two-stage classifier inference. Used when the static
    filter ran and we need its post-filter view of detections.
    """
    init_csv(output_csv, csv_fields)
    cam_prefix, camera_path = _camera_prefix_for(image_dir)
    total_paths = len(paths)

    for idx_p, path in enumerate(paths):
        path_str = str(path)
        rel_id   = _rel_id_for(path, image_dir)
        detections = filtered_full.get(path_str, [])

        # Avoid the cv2.imread when there's nothing to crop and no overlay
        # to draw; metadata-only CSV row is fine.
        skip_image_read = (
            not detections
            and not cfg.get("debug_save_empty_frames", False)
        )
        image = None if skip_image_read else cv2.imread(path_str)

        _emit_frame_results(
            idx_p, total_paths, rel_id, path_str, image,
            detections, strip_temps,
            cam_prefix, camera_path, zone,
            output_csv, cfg, csv_fields, crop_dir,
            model,
            debug, debug_dir,
        )


def main(image_dir, output_csv="results.csv", debug=False,
         manual_roi=None, config=None, debug_csv=False,
         crop_dir=None, debug_dir=None):
    """
    Main pipeline — orchestrates all steps.

    Args:
        image_dir:  path to folder containing images for one plot
        output_csv: path to output CSV file
        debug:      save intermediate debug images to debug_dir
        manual_roi: manually draw ROI instead of using green marker
        config:     dict of parameter overrides merged with DEFAULT_CONFIG
        debug_csv:  False = clean CSV for Maria (default)
                    True  = full debug CSV with bbox, binary/group confidence
        crop_dir:   directory for detection crops (default: <output_csv parent>/crops)
        debug_dir:  directory for debug images  (default: <output_csv parent>/debug)
    """
    cfg        = {**DEFAULT_CONFIG, **(config or {})}
    csv_fields = CSV_FIELDS_DEBUG if debug_csv else CSV_FIELDS_MARIA

    image_dir  = Path(image_dir)
    output_csv = Path(output_csv)
    out_base  = output_csv.parent if output_csv.parent != Path('.') else PROJECT_DIR

    crop_dir  = Path(crop_dir)  if crop_dir  else out_base / "crops"
    debug_dir = Path(debug_dir) if debug_dir else out_base / "debug"
    crop_dir.mkdir(parents=True, exist_ok=True)

    # Sort by filename for time-order; use relative path as unique key to handle
    # duplicate filenames across subfolders
    all_found = list(image_dir.rglob("*.JPG")) + list(image_dir.rglob("*.jpg"))
    # Default: cheap filename-numeric sort. Opt in to EXIF sort via
    # cfg["use_exif_sort"] when filenames may not reflect capture order.
    sort_key = robust_sort_key if cfg.get("use_exif_sort", False) else filename_sort_key
    paths = sorted(all_found, key=sort_key)

    # Optional manual triage: read a flag file (e.g. produced by feh-triage.sh)
    # and reduce paths to those entries plus ±frames_context neighbours.
    # Off by default — only kicks in when frames_from is set.
    frames_from = cfg.get("frames_from")
    if frames_from:
        flag_path = Path(frames_from)
        if not flag_path.is_absolute():
            flag_path = image_dir / flag_path
        if flag_path.exists():
            wanted = set()
            for line in flag_path.read_text(encoding="utf-8").splitlines():
                line = line.strip()
                if not line:
                    continue
                try:
                    wanted.add(Path(line).resolve())
                except OSError:
                    continue
            ctx = max(0, int(cfg.get("frames_context", 0)))
            keep_idx = set()
            for i, p in enumerate(paths):
                try:
                    rp = p.resolve()
                except OSError:
                    rp = p
                if rp in wanted:
                    for j in range(max(0, i - ctx), min(len(paths), i + ctx + 1)):
                        keep_idx.add(j)
            if keep_idx:
                paths = [paths[i] for i in sorted(keep_idx)]
                print(f"frames_from={flag_path.name}: {len(wanted)} flagged, {len(paths)} processed (with ±{ctx} context)")
            else:
                print(f"frames_from={flag_path}: no matches in image_dir, processing all images")
        else:
            print(f"frames_from={flag_path} not found; processing all images")
    print(f"First 3 frames (sorted): {[p.name for p in paths[:3]]}")
    # Warn about duplicate filenames
    from collections import Counter
    name_counts = Counter(p.name for p in paths)
    dups = {k: v for k, v in name_counts.items() if v > 1}
    if dups:
        print(f"WARNING: {len(dups)} duplicate filename(s) found across subfolders.")
        print("Using relative path as unique ID in CSV and debug outputs.")
    if not paths:
        print(f"No images found in {image_dir}")
        return
    print(f"Found {len(paths)} images")

    if debug:
        debug_dir.mkdir(parents=True, exist_ok=True)
        print(f"Debug images will be saved to {debug_dir}/")

    # Use manual_roi arg if explicitly passed, otherwise fall back to config
    use_manual_roi = manual_roi if manual_roi is not None else cfg.get("manual_roi", False)
    zone, _ = setup_roi(paths, cfg, use_manual_roi)

    background = build_background(paths, cfg)
    if background is None and cfg.get("rolling_window", 0) == 0:
        print("Failed to build background.")
        return

    if cfg.get("skip_classification", False):
        model = None
        print("skip_classification=True — running detection only, no classifier")
    else:
        binary_model, binary_tf, group_model, group_tf, group_classes, clf_device = load_model_and_classes()
        model = (binary_model, binary_tf, group_model, group_tf, group_classes, clf_device)

    use_static_filter = (
        cfg.get("enable_static_filter", True)
        and cfg.get("static_max_frames", 0) > 0
    )

    if use_static_filter:
        # Two-pass: detect everything, run global static filter, then crop.
        all_detections, strip_temps = detect_all_frames(
            paths, background, zone, cfg, debug, debug_dir, image_dir=image_dir,
        )
        print(f"Before filter: {sum(len(v) for v in all_detections.values())} detections across {len(all_detections)} frames")
        filtered = filter_static_detections(all_detections, cfg, paths=paths)
        print(f"After filter:  {sum(len(v) for v in filtered.values())} detections across {len(filtered)} frames")

        classify_and_write(
            paths, filtered, zone, model,
            output_csv, cfg, csv_fields,
            crop_dir, debug, debug_dir, image_dir=image_dir,
            strip_temps=strip_temps,
        )
    else:
        # Single-pass: no static filter means no global decision step, so we
        # can detect and save crops in one loop and skip the second imread.
        print("Static filter disabled — running single-pass detect+classify")
        detect_and_classify_streaming(
            paths, zone, model,
            output_csv, cfg, csv_fields, crop_dir,
            background=background, debug=debug, debug_dir=debug_dir,
            image_dir=image_dir,
        )

    print(f"\nDone. Results saved to {output_csv}")

### Batch Run

Process all leaf folders under a root directory. One CSV per folder saved to `results/`.

1. Set `ROOT` to your data directory
2. Run the preview cell to confirm folders
3. Run the execute cell to process all folders

### Run

Use default config, or override specific parameters via `config={}`. Only override what
need to change.

In [7]:
# ══════════════════════════════════════════════════════════════════
# BATCH RUN — one CSV per leaf folder
# ══════════════════════════════════════════════════════════════════
#
# A "leaf folder" is any directory that contains images and no subdirectories.
#
# Each leaf folder contains images from one camera at one fixed position
# covering one plot. Background subtraction is therefore valid within a
# single leaf folder, as the background is consistent across all frames.
#
# Folder structure example:
#   ROOT/
#   ├── hdd1_cg_asa_p1/
#   │   ├── 101_wsct/   ← one camera, one plot  →  hdd1_cg_asa_p1_101_wsct.csv
#   │   └── 102_wsct/   ← one camera, one plot  →  hdd1_cg_asa_p1_102_wsct.csv
#   └── hdd1_enbranten_aral_p1/
#       ├── 101_wsct/   ← one camera, one plot  →  hdd1_enbranten_aral_p1_101_wsct.csv
#       └── 102_wsct/   ← one camera, one plot  →  hdd1_enbranten_aral_p1_102_wsct.csv

from pathlib import Path

# ── Configure ─────────────────────────────────────────────────────
ROOT        = Path(IMAGE_FOLDER) # Change this in first cell

# Results saved alongside Pollinators folder (not inside it):
#   Insects_images/
#   ├── Pollinators/    ← image data (ROOT)
#   ├── results/        ← first run
#   ├── results_1/      ← second run (auto-incremented)
#   └── results_2/      ← third run
def make_results_dir(base: Path) -> Path:
    """Create results folder next to Pollinators, auto-increment if already exists."""
    candidate = base / "results"
    if not candidate.exists():
        candidate.mkdir(parents=True)
        return candidate
    i = 1
    while True:
        candidate = base / f"results_{i}"
        if not candidate.exists():
            candidate.mkdir(parents=True)
            return candidate
        i += 1

RESULTS_DIR = make_results_dir(ROOT.parent)
print(f"Results will be saved to: {RESULTS_DIR}")

BATCH_CONFIG = {
    "use_roi": True,   # Set False to skip ROI setup and use full image for detection
    "skip_insectnet": True,   # Set False for full pipeline
    "rolling_window": 1,
    "debug_outputs": "annotate",      # only _4_final_saved_crops + _4_offset.json
    "debug_save_empty_frames": False,  # annotate.py only needs frames that have crops
    "debug_max_width": 1024,           # shrink debug images; crop files stay original quality
    "debug_jpeg_quality": 60,
}

# ── Find all leaf folders ──────────────────────────────────────────
def get_leaf_dirs(root):
    """Return all leaf directories (no subdirectories) that contain images.

    If `root` itself is a leaf folder (has images, no subdirs), return [root].
    Otherwise recurse and return every descendant leaf folder with images.

    Each leaf folder corresponds to one camera at one fixed position covering
    one plot. Images within a leaf folder share a consistent background,
    making background subtraction valid within each folder.
    """
    has_subdirs = any(p.is_dir() for p in root.iterdir())
    has_images  = any(root.glob("*.JPG")) or any(root.glob("*.jpg"))
    if has_images and not has_subdirs:
        return [root]

    leaf = []
    for d in sorted(root.rglob("*")):
        if d.is_dir() and not any(x.is_dir() for x in d.iterdir()):
            if any(d.glob("*.JPG")) or any(d.glob("*.jpg")):
                leaf.append(d)
    return leaf

leaf_dirs = get_leaf_dirs(ROOT)

# ── Preview ───────────────────────────────────────────────────────
print(f"Found {len(leaf_dirs)} leaf folder(s):")
for d in leaf_dirs:
    n = len(list(d.glob("*.JPG"))) + len(list(d.glob("*.jpg")))
    csv_name = f"{d.parent.name}_{d.name}.csv"
    rel = d.relative_to(ROOT) if d != ROOT else d.name
    print(f"  {rel}  →  {n} images  →  {csv_name}")


Results will be saved to: Insects_images/results_2
Found 1 leaf folder(s):
  hdd_1_2025_skredet_vau_p2_20250708/Vau_p2_20250708/102_WSCT  →  9133 images  →  Vau_p2_20250708_102_WSCT.csv


In [8]:
# Default run
#main(
 #   image_dir  = PROJECT_DIR / "Insects_images",
  #  output_csv = "results.csv",
   # debug      = True,
    #manual_roi = False,
    #config = {
    #    "marker_hue":      (80, 100),
    #    "marker_sat_min":  100,
    #    "marker_val_min":  80,
    #    "marker_min_area": 50,
    #}
#)

# Example: override specific parameters after tuning
# main(
#     image_dir  = PROJECT_DIR / "images",
#     output_csv = "results.csv",
#     config = {
#         "sunny_shutter_threshold": 150,   # adjust after checking cloudy images
#         "static_max_frames":       10,    # tighten if too many false positives
#         "darker_threshold":        30,    # lower = more sensitive
#         "background_sample_size":  100,   # reduce if memory is tight
#     }
# )

In [9]:
# ── Run batch ─────────────────────────────────────────────────────
# Run the preview cell first to confirm the folder list looks correct.

for camera_dir in leaf_dirs:
    n = len(list(camera_dir.glob("*.JPG"))) + len(list(camera_dir.glob("*.jpg")))
    if n == 0:
        print(f"Skipping {camera_dir} — no images")
        continue

    # One output folder per camera: results/hdd1_cg_asa_p1_101_wsct/
    csv_name   = f"{camera_dir.parent.name}_{camera_dir.name}"
    camera_out = RESULTS_DIR / csv_name
    camera_out.mkdir(exist_ok=True)

    print(f"\n{'='*60}")
    print(f"Camera : {camera_dir.relative_to(ROOT)}")
    print(f"Images : {n}")
    print(f"Output : {camera_out}/")
    print(f"{'='*60}")

    try:
        main(
            image_dir  = camera_dir,
            output_csv = str(camera_out / "results.csv"),
            crop_dir   = str(camera_out / "crops"),
            debug_dir  = str(camera_out / "debug"),
            debug      = True,
            debug_csv  = True,   # use full CSV with bbox coords for annotation tool
            config     = BATCH_CONFIG,
        )
        print(f"\u2713 Done — {csv_name}/")
    except KeyboardInterrupt:
        raise
    except Exception as e:
        import traceback
        print(f"\u2717 ERROR in {camera_dir.name}: {e}")
        traceback.print_exc()
        continue

print(f"\nAll done. CSVs saved under: {RESULTS_DIR}")



Camera : hdd_1_2025_skredet_vau_p2_20250708/Vau_p2_20250708/102_WSCT
Images : 9133
Output : Insects_images/results_2/Vau_p2_20250708_102_WSCT/
First 3 frames (sorted): ['WSCT0001.JPG', 'WSCT0002.JPG', 'WSCT0003.JPG']
Found 9133 images
Debug images will be saved to Insects_images/results_2/Vau_p2_20250708_102_WSCT/debug/
Marker found at (2386, 1158)
Watching zone covers 32.8% of image
background_sample_size=0 — skipping global background (rolling only)
Loading classifiers...
Binary classifier loaded: efficientnet | img=256px | device=cpu
Group classifier loaded: insectnet | classes=['bumblebee', 'fly', 'butterfly_moth', 'other'] | img=224px
Classifiers loaded.

Static filter disabled — running single-pass detect+classify
Detecting + classifying visitors (single-pass)...
  [1/9133] WSCT0001.JPG | sunny | sharpness=400 | no detection
  [2/9133] WSCT0002.JPG | sunny | sharpness=403 | 2 candidate(s)
  [3/9133] WSCT0003.JPG | sunny | sharpness=396 | 2 candidate(s)
  WSCT0003.JPG [roi]: bina